In [ ]:
# --- make the safety_probes package importable ---
# Preferred: run `pip install -e .` from the repo root once; then this cell is a no-op.
# Fallback (below) handles a not-yet-installed checkout, including VSCode kernels whose working
# directory is the workspace root rather than this notebook's folder.
import sys, pathlib
try:
    import safety_probes  # already installed (editable) -> nothing to do
except ModuleNotFoundError:
    starts = []
    _f = globals().get("__vsc_ipynb_file__")          # VSCode injects the notebook's own path
    if _f:
        starts.append(pathlib.Path(_f).resolve())
    starts.append(pathlib.Path.cwd())                 # classic Jupyter: cwd = notebook dir
    for _s in starts:
        _root = next((p for p in [_s, *_s.parents]
                      if (p / "safety_probes" / "__init__.py").exists()), None)
        if _root:
            sys.path.insert(0, str(_root)); break
    import safety_probes
print("safety_probes:", safety_probes.__file__)

# High-stakes — detection & steering (parity driver)

Reproduces **§3** of `safety_probes.ipynb` using the `safety_probes` package. Positive class =
**high-stakes** (1), negative = low-stakes (0). Same pipeline and section structure as the deception
notebook (§§1–8 mirror it cell-for-cell), run with the high-stakes spec, `content_tokens` mask
policy, and MUP data. Audited choices: **detection layer 20**, steering block **[9, 10, 11, 12, 13]**.

`extra["raw"]` on each high-stakes Split carries the raw (non-templated) scenario text per example,
index-aligned — the readout and steering want plain text. (The notebook read these inline as
`hs_all_texts[i] for i in train_idx`; the loader now exposes them, see datasets M6/M8.)

In [ ]:
from safety_probes import runtime, datasets, metrics, steering, viz, evaluation, export
from safety_probes.chat import chat_text
from safety_probes.datasets import mup_to_messages
from safety_probes.extraction import get_pooled, extract_full_sequence_one_layer
from safety_probes.metrics import (cv_auroc_by_layer, build_probe_grid, evaluate_grid, pack)
from safety_probes.probes import MMProbe, LRProbe
from safety_probes.steering import (nie_layer_sweep, pick_block, build_directions, calibrate_block)
from safety_probes.evaluation import steering_nie_llm_evaluation
from safety_probes.judge import N_JUDGE_PROMPTS
from safety_probes.viz import (pca_and_projection_panel, plot_roc_curves, probe_direction_cosines,
                               show_token_heatmaps, show_attention_heatmaps, plot_steering_effects,
                               show_qualitative)
from safety_probes.spec import BehaviourSpec
from safety_probes.datasets import HS_MAX_LEN
from sklearn.metrics import roc_auc_score
import numpy as np, torch as t
import plotly.graph_objects as go

# Crisp inline figures on hi-dpi screens (pairs with viz._use_style()'s DPI bump — plan #50/M15).
%config InlineBackend.figure_format = "retina"

runtime.init()

## 1. BehaviourSpec and Datasets

The spec configures the NIE readout (few-shot + YES/NO targets), the judge rubric, and the steering
coefficients; the loaders pull the pinned MUP data (train/test + Anthropic-HH OOD).

In [ ]:
# Spec moved to the package (plan #48) so the comparison notebook shares one source of truth —
# content unchanged (incl. the CAA-style rubric, #47); see safety_probes/behaviours.py.
from safety_probes.behaviours import HIGH_STAKES_SPEC
HIGH_STAKES_SPEC

In [ ]:
# --- High-stakes: models-under-pressure synthetic train set [HS §2.4] ---
hs = datasets.load_high_stakes()               # .train / .test  (each Split has extra["raw"])
# --- High-stakes OOD: anthropic_hh_balanced [HS Table 2]. ToolACE removed (decision 2026-07-19,
# plan #34): tool-calling dialogues are a different interaction modality, out of scope. ---
# Real-world, human-generated -> genuinely OOD vs the synthetic train set.
hs_ood = datasets.load_high_stakes_ood()       # {"anthropic_hh_balanced": Split(extra=raw,msgs)}

## 2. Extracting Activations

In [ ]:
# High-stakes pooled activations (full-sequence mask); HS_MAX_LEN=256 matches [NB].
hs_train_mean, hs_train_last = get_pooled("hs_train", hs.train.texts, hs.train.masks, max_length=HS_MAX_LEN)
hs_test_mean,  hs_test_last  = get_pooled("hs_test",  hs.test.texts,  hs.test.masks,  max_length=HS_MAX_LEN)
hs_ood_pooled = {}
for cfg, split in hs_ood.items():
    m, l = get_pooled(f"hs_ood_{cfg}", split.texts, split.masks, max_length=HS_MAX_LEN)
    hs_ood_pooled[cfg] = (m, l)
print("High-stakes pooled activations:", tuple(hs_train_mean.shape))

## 3. Layer Selection

Same two purpose-matched criteria as the deception notebook §3 (self-contained recap, since the
notebooks now live apart):

- **Detection layer** = argmax **pair-aware 5-fold CV AUROC** of an MM probe on mean-pooled
  activations, over all layers ("pair" here = the MUP `pair_id` group: all tone/language/style
  variants of one high/low situation pair stay on one side of every fold). Cheap because
  activations are cached at every layer. LR curve overlaid as the proxy-assumption check.
- **Steering block** (chosen in §7a, once the readout exists) = contiguous layers with high
  **NIE** — the causal analogue, [GoT §3]'s "group (b)".

Prior expectation from the papers: mid-depth, ≈ layers 12–16 for an 8B ([HS §3] landed at 39% depth
on a 70B; [NB] §5 used 16). The audited run's argmax was deeper: **20**. Expect a broader,
sub-ceiling curve here, unlike deception's ceiling plateau — high-stakes is content-defined (no
persona shortcut), so the CV AUROC is a real difficulty estimate, not a sanity check.

**EDIT `HS_DETECT_LAYER`** in the cell below (audited: 20).

In [ ]:
hs_cv_mean, hs_cv_std = cv_auroc_by_layer(hs_train_mean, hs.train.labels, hs.train.pairs)
# LR overlay (robustness, decision 2026-07-11): the grid's LR probes are trained at the chosen layer
# too, so check LR's curve tracks MM's before committing. MM stays the selector. (~160 LR fits;
# noticeably slower than the closed-form MM sweep.)
hs_cv_lr_mean, hs_cv_lr_std = cv_auroc_by_layer(hs_train_mean, hs.train.labels, hs.train.pairs, probe_cls=LRProbe)

# ===== EDIT HERE: pick the detection layer from the plot below. =====
# argmax of the CV-AUROC curve is only a SUGGESTED DEFAULT. Read the curve (and the NIE bars in the
# steering subsection) and set the constant; everything downstream keys off it.
print(f"Suggested (argmax CV AUROC): {int(np.argmax(hs_cv_mean))}")
HS_DETECT_LAYER = int(np.argmax(hs_cv_mean))   # <-- override with your choice (audited notebook = 20)
# ====================================================================

viz.plot_layer_selection(hs_cv_mean, hs_cv_std, hs_cv_lr_mean, hs_cv_lr_std, HS_DETECT_LAYER, "High-stakes")

## 4. PCA Visualization

[GoT §4] visualizes truth with PCA on **centered** activations. Same panels as deception §4, same
caveat: PCA shows *variance*, and class separation can ride a topic confound rather than the
behaviour ([Apollo §4.1.2]). The real evidence a linear probe exists is the histogram of projections
**onto the trained probe direction** — shown alongside. One difference from deception to expect:
high-stakes classes differ in *content* (not just a system line), so some PCA separation here would
not be surprising — which is exactly why the projection histogram, not the PCA scatter, stays the
load-bearing panel.

In [ ]:
hs_mm_detect = MMProbe.from_data(hs_train_mean[:, HS_DETECT_LAYER, :], hs.train.labels)
pca_and_projection_panel(hs_train_mean, hs.train.labels, HS_DETECT_LAYER, hs_mm_detect.direction, "High-stakes")

## 5. Probing

Full-sequence activations at `HS_DETECT_LAYER` (for the attention probe + per-token LR training,
truncated at `HS_MAX_LEN=256` as in [NB] §5; the Anthropic-HH full-seq tensors are RAM-only,
`use_cache=False`, plan #24), then the 5-probe grid.

In [ ]:
# Full-sequence activations at the chosen detection layer, for the attention probe and
# per-token LR training (extract_full_sequence_one_layer is defined in §1c / extraction.py).
hs_train_full, hs_train_fmask = extract_full_sequence_one_layer("hs_train", hs.train.texts, hs.train.masks, HS_DETECT_LAYER, HS_MAX_LEN)
hs_test_full,  hs_test_fmask  = extract_full_sequence_one_layer("hs_test",  hs.test.texts,  hs.test.masks,  HS_DETECT_LAYER, HS_MAX_LEN)
hs_ood_full = {}
for cfg, split in hs_ood.items():
    hs_ood_full[cfg] = extract_full_sequence_one_layer(f"hs_ood_{cfg}", split.texts, split.masks, HS_DETECT_LAYER, HS_MAX_LEN, use_cache=False)


In [ ]:
# --- High-stakes grid ---
Lh = HS_DETECT_LAYER
hs_grid = build_probe_grid(hs_train_mean[:, Lh, :], hs_train_last[:, Lh, :], hs_train_fmask,
                           hs_train_full, hs.train.masks, hs.train.labels)

### 5.i. Probing Evaluations

ID test + the real-world OOD set (Anthropic HH; ToolACE removed, plan #34): AUROC (bootstrap 95% CI)
+ TPR@1%FPR. Both OOD conversations and ID scenarios use the same `content_tokens` mask.

In [ ]:
hs_eval_packs = {"test (ID)": pack(hs_test_mean[:, Lh, :], hs_test_last[:, Lh, :], hs_test_full, hs_test_fmask, hs.test.labels)}
for cfg, split in hs_ood.items():
    m, l = hs_ood_pooled[cfg]; full, fmask = hs_ood_full[cfg]
    hs_eval_packs[f"{cfg} (OOD)"] = pack(m[:, Lh, :], l[:, Lh, :], full, fmask, split.labels)
print("=== High-stakes probe grid (detection layer", Lh, ") ===")
hs_grid_df = evaluate_grid(hs_grid, hs_eval_packs)
display(hs_grid_df)

# [HS Fig. 2] reports variance across SEEDS (training stability); the table's CI is a bootstrap over
# EXAMPLES (measurement uncertainty of a finite eval set). The grid ensembles the 3 seeds, which
# hides training stability - so print the per-seed test AUROCs here to make it visible.
with t.no_grad():
    _seed_aurocs = [roc_auc_score(hs.test.labels.numpy(), p(hs_test_full, hs_test_fmask).sigmoid().numpy())
                    for p in hs_grid["LR-attention"][0]]
print("LR-attention per-seed test AUROC (training stability):", [f"{a:.3f}" for a in _seed_aurocs])

plot_roc_curves(hs_grid, hs_eval_packs["test (ID)"], f"ROC curves - high-stakes ID test (layer {Lh}, {len(hs.test.labels)} examples)")
plot_roc_curves(hs_grid, hs_eval_packs["anthropic_hh_balanced (OOD)"], "ROC curves - high-stakes OOD: Anthropic HH")

In [ ]:
# Direction geometry, as §2d. Compare the pattern across behaviours by eye.
_ = probe_direction_cosines(hs_grid, hs_train_mean[:, Lh, :], hs_train_last[:, Lh, :], hs.train.labels, f"High-stakes (detection layer {Lh})")

### 5.ii. Probing Verdict

Best **reader** per eval set — AUROC point estimates from the grid above — and the LR-family win
share: the READ half of the headline hypothesis ([Apollo App. D.1]: LR reads best). The write half
lives at the end of the steering section (§7d). (Formerly computed cross-behaviour in
`summary.ipynb`, removed 2026-07-30 — plan #45.)

In [ ]:
# --- Probing verdict: who READS high-stakes best? ---
# AUROC point estimates parsed from hs_grid_df above (CIs stripped; TPR@1%FPR is reported in the
# table but plays no role in the verdict). idxmax per eval set = best reader.
from safety_probes import summary
hs_read_pivot = summary.auroc_pivot(hs_grid_df, "high-stakes")
display(hs_read_pivot.round(3))
hs_best_readers = hs_read_pivot.idxmax()
print("Best probe per eval set:"); print(hs_best_readers.to_string())
hs_lr_share = hs_best_readers.str.startswith("LR").mean()

In [ ]:
print(f"\nLR-family wins {hs_lr_share:.0%} of high-stakes detection columns "
      "([Apollo App. D.1] predicts LR reads best).")

## 6. Sample Token Heatmaps

Because `LR (mean)` is trained per-token, each token has its own score = projection of that token's
activation onto the probe direction. Colouring the tokens shows *where in the text* the probe fires —
[HS Fig. 7]'s analogue (high-stakes lights up on the words that make the request dangerous). A second
panel (§6b) shows the **attention probe's weights** — "where the best detector looks" next to "what
the LR direction scores" — on the same ID **and** OOD examples.

Reading the render (same conventions as deception §6):

- **Token color** — each token's activation projected onto the probe direction, then **standardized** (centered on the shown examples' mean probed token, scaled by this example's spread): red = above that average (toward the high-stakes direction), blue = below, white = at the mean. **Relative shading within these examples for contrast, not a per-token verdict.**
- **High-stakes score** (header) — the example's **raw** mean projection over its probed tokens. **Uncalibrated**: not a probability, no fixed range, sign not meaningful on its own — compare across examples only. The *ground-truth label* beside it is the dataset's true class, not a prediction.

### 6a. Probe Heatmaps

In [ ]:
hs_lr_mean_dir = hs_grid["LR (mean)"][0].direction
show_token_heatmaps(hs.test.texts, hs.test.masks, hs.test.labels, hs_lr_mean_dir, HS_DETECT_LAYER, "High-stakes ID test", "high-stakes", "low-stakes", score_name="high-stakes score")
for cfg, split in hs_ood.items():
    show_token_heatmaps(split.texts, split.masks, split.labels, hs_lr_mean_dir, HS_DETECT_LAYER, f"{cfg} (OOD)", "high-stakes", "low-stakes", score_name="high-stakes score")

### 6b. Attention Heatmaps

The attention probe scores each token's relevance with its learned query matrix, softmaxes those
scores into weights α over the probed tokens, pools the tokens by that weighted sum, and classifies
the pooled vector — so the highlighted weights below are exactly the share of the decision each
token contributes (full math: deception notebook §6b).

**Known expected finding for high-stakes** (plan #25): attention mass parking on BOS/template
tokens (attention-sink behaviour) is a *result*, not a rendering bug — note it, don't debug it.

In [ ]:
# Attention-weights view; watch for attention mass on BOS/template tokens (see helper docstring).
hs_attn_seeds = hs_grid["LR-attention"][0]
show_attention_heatmaps(hs_test_full, hs_test_fmask, hs.test.texts, hs.test.labels, hs_attn_seeds, "High-stakes ID", "high-stakes", "low-stakes")
# OOD attention view: same configs as the LR-score panels above; OOD full-seq tensors are RAM-only.
for cfg, split in hs_ood.items():
    full, fmask = hs_ood_full[cfg]
    show_attention_heatmaps(full, fmask, split.texts, split.labels, hs_attn_seeds, f"{cfg} (OOD)", "high-stakes", "low-stakes")

## 7. Steering

**No paper and no [NB] section steers high-stakes — this is the novel contribution.** Machinery is
identical to the deception notebook §7 (per-layer refit + mean-gap-calibrated MM / LR / LR-attention
directions, added at every token position across the block).

### 7a. Steering-Block Selection (NIE sweep + baseline gate)

For each layer, add the calibrated MM direction (coef +1) at that single layer during the readout on
a small validation subset, and measure the NIE. The contiguous high-NIE band becomes the steering
block ([GoT §3] group (b)). First we compute the baseline `PD⁺ − PD⁻` gate.

Same two scope notes as deception §7a: the sweep subset comes from the **train** split (block
selection never touches the test items the ID dose-response in §7c evaluates), and the block is
selected with the **MM** direction — a mild, disclosed pro-MM thumb on the scale; a shared block
keeps the §7c head-to-head fair at intervention time.

**EDIT `HS_STEER_BLOCK`** in the pick cell below (audited: [9, 10, 11, 12, 13]).

In [ ]:
# Sweep subset from the TRAIN split (see §7a note); raw scenario texts, as the readout wants plain text.
# raw scenario texts by class, from the Split's index-aligned extra["raw"] (see header note)
hs_train_raw, hs_train_lab = hs.train.extra["raw"], hs.train.labels.tolist()
hs_test_raw,  hs_test_lab  = hs.test.extra["raw"],  hs.test.labels.tolist()
hs_sweep_pos_tx = [r for r, l in zip(hs_train_raw, hs_train_lab) if l == 1][:32]
hs_sweep_neg_tx = [r for r, l in zip(hs_train_raw, hs_train_lab) if l == 0][:32]

hs_nie, hs_gap = nie_layer_sweep(HIGH_STAKES_SPEC, hs_train_mean, hs.train.labels, hs_sweep_pos_tx, hs_sweep_neg_tx)

In [ ]:
# ===== EDIT HERE: pick the high-stakes steering block from the NIE bars below. =====
# pick_block returns the highest-summed contiguous window as a SUGGESTED DEFAULT. Read the bar chart
# and set the list to your choice (any layers you want to steer).
print(f"Suggested (top NIE window): {pick_block(hs_nie)}")
HS_STEER_BLOCK = pick_block(hs_nie)   # <-- override, e.g. [9,10,11,12,13] (audited notebook)
# =======================================================================

viz.plot_nie_bars(hs_nie, HS_STEER_BLOCK, "High-stakes")

### 7b. Directions & Calibration

MM / LR / LR-attention, each refit and mean-gap-calibrated per layer of the block. This is the
longest cell of the notebook: LR-attention trains 3-seed probes at every block layer on full-sequence
train activations (~8–12 min per layer for high-stakes; the tensors are RAM-only, plan #24/#27).

In [ ]:
hs_dirs = calibrate_block(
    build_directions(HIGH_STAKES_SPEC, HS_STEER_BLOCK, hs_train_mean, hs.train.labels, hs.train.texts, hs.train.masks, HS_MAX_LEN),
    hs_train_mean, hs.train.labels)
print(f"High-stakes steering directions calibrated: {list(hs_dirs)} on block {HS_STEER_BLOCK}")

### 7c. Steering Evaluations

ID first (positive control), then OOD (the claim — Anthropic HH). NIE dose-response + LLM-judged
generations. Unlike deception, high-stakes ID **keeps its judged-generation arm**: its
`N_JUDGE_PROMPTS` prompts are distinct low-stakes test scenarios (no byte-identical-prompt
degeneracy, no target-naming persona line — the target is never stated in the prompt).

In [ ]:
# --- High-stakes steering: ID then OOD ---
# ID readout/eval transcripts from the TEST split - disjoint from the train-split sweep subset (§7a).
hs_val_pos_tx = [r for r, l in zip(hs_test_raw, hs_test_lab) if l == 1][:32]
hs_val_neg_tx = [r for r, l in zip(hs_test_raw, hs_test_lab) if l == 0][:32]
hs_id_prompts = [chat_text([{"role": "user", "content": hs_val_neg_tx[i]}], add_generation_prompt=True) for i in range(N_JUDGE_PROMPTS)]
print("=== High-stakes steering: IN-DISTRIBUTION (positive control) ===")
hs_id_results = steering_nie_llm_evaluation(HIGH_STAKES_SPEC, hs_dirs, hs_id_prompts, hs_val_neg_tx[:32], hs_val_pos_tx[:32], list(HIGH_STAKES_SPEC.steer_coefs))
plot_steering_effects(hs_id_results, "High-stakes steering (ID)")

hh = hs_ood["anthropic_hh_balanced"]
hh_raw, hh_lab, hh_msgs = hh.extra["raw"], hh.labels.tolist(), hh.extra["msgs"]
hs_ood_neg = [x for x, l in zip(hh_raw, hh_lab) if l == 0][:32]   # plaintext, for the readout
hs_ood_pos = [x for x, l in zip(hh_raw, hh_lab) if l == 1][:32]
# Generation prompts: the parsed conversation (up to its last USER turn) + generation prompt, so the
# model continues the real dialogue rather than reading a JSON blob inside a user turn.
def trim_to_last_user(msgs):
    last_user = max((i for i, m in enumerate(msgs) if m["role"] == "user"), default=None)
    return msgs[: last_user + 1] if last_user is not None else msgs
hs_ood_neg_msgs = [m for m, l in zip(hh_msgs, hh_lab) if l == 0]
hs_ood_prompts = [chat_text(trim_to_last_user(m), add_generation_prompt=True) for m in hs_ood_neg_msgs[:N_JUDGE_PROMPTS]]
print("=== High-stakes steering: OUT-OF-DISTRIBUTION (Anthropic HH) ===")
hs_ood_results = steering_nie_llm_evaluation(HIGH_STAKES_SPEC, hs_dirs, hs_ood_prompts, hs_ood_neg, hs_ood_pos, list(HIGH_STAKES_SPEC.steer_coefs))
plot_steering_effects(hs_ood_results, "High-stakes steering (OOD: Anthropic HH)")

In [ ]:
show_qualitative(HIGH_STAKES_SPEC, hs_dirs, hs_id_prompts[0])

### 7d. Steering Verdict

Best **writer** per eval set — NIE at calibrated coef +1 — and the MM win share: the WRITE half of
the headline hypothesis ([GoT Table 2]: MM writes best), plus the per-behaviour read/write
dissociation line. The judge feeds no number here; its coherence panels above discipline the NIE
wins. (Formerly computed cross-behaviour in `summary.ipynb`, removed 2026-07-30 — plan #45.)

In [ ]:
# --- Steering verdict: who WRITES high-stakes best? ---
# NIE at calibrated coef +1 (= push one full class-mean gap) per direction × eval set; idxmax per
# eval set = best writer. The judge feeds no number here — check the coherence panels above before
# trusting any win (an NIE shift at collapsed coherence is not steering).
import pandas as pd
from safety_probes import summary
hs_steer_by_evalset = {"ID": hs_id_results, "Anthropic HH (OOD)": hs_ood_results}
hs_write_tbl = pd.DataFrame({f"high-stakes: {k}": summary.nie_at(v) for k, v in hs_steer_by_evalset.items()})
display(hs_write_tbl.round(3))
hs_best_writers = hs_write_tbl.idxmax()
print("Best direction per eval set:"); print(hs_best_writers.to_string())
hs_mm_share = (hs_best_writers == "MM").mean()

In [ ]:
print(f"\nMM wins {hs_mm_share:.0%} of high-stakes steering columns ([GoT Table 2] predicts MM writes best).")
print("Read/write dissociation " + ("HOLDS" if hs_lr_share > 0.5 and hs_mm_share > 0.5 else "does NOT clearly hold")
      + " for high-stakes on this run (LR reads best + MM writes best; judge-coherence caveat applies).")

## 8. Export bundle

In [ ]:
# Saving Probe
export.export_behaviour(
    HIGH_STAKES_SPEC, hs_grid, HS_DETECT_LAYER, HS_STEER_BLOCK, hs_dirs,
    d_model=hs_train_mean.shape[-1],
    notes="OOD: anthropic_hh_balanced only (ToolACE removed, plan #34).")

# Freeze the final numbers as artifacts in exported_probes/ — CSVs for thesis tables, and the FULL
# dose-response JSON for the cross-behaviour comparison notebook (plan #48). Verdicts themselves
# are computed inline in §5.ii and §7d.
from safety_probes import summary
summary.save_detection(hs_grid_df, "high-stakes")
summary.save_steering(hs_steer_by_evalset, "high-stakes")
summary.save_steering_full(hs_steer_by_evalset, "high-stakes")